# City-wide Forecast Backtesting

This notebook tests the selected model under realistic forecasting modes: **daily-updated one-day-ahead** versus **fully recursive long-horizon** forecasts.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.citywide_forecasting import (
    daily_updated_predictions,
    fit_nb,
    forecast_metrics,
    hybrid_forecast,
    prepare_actual_year,
    prepare_daily_data,
    recursive_backtest,
    recursive_forecast,
)

DATA_PATH = PROJECT_ROOT / "data" / "requests.csv"
ACTUAL_2026_PATH = PROJECT_ROOT / "data" / "2026requests.csv"

ISSUE_TYPE = "garbage"
SELECTED_MODEL = "nb_recent3_lag7"
INCLUDE_HOLIDAYS = True
TREND_MODE = "log"
LOG_TREND_DIVISOR = 3

# Historical long-horizon check.
BACKTEST_YEAR = 2025
BACKTEST_TREND_MODES = ["linear", "log"]

# 2026 test / planning horizon.
FORECAST_YEAR = 2026
FORECAST_END = "2026-12-31"

## 2. Prepare 2020–2025 history and available 2026 actuals

In [2]:
raw_history = pd.read_csv(DATA_PATH)
raw_2026 = pd.read_csv(ACTUAL_2026_PATH)

history = prepare_daily_data(
    raw_history,
    issue_type=ISSUE_TYPE,
    start_year=2020,
    end_year=2025,
    log_divisor=LOG_TREND_DIVISOR,
)

actual_2026 = prepare_actual_year(
    raw_2026,
    FORECAST_YEAR,
    issue_type=ISSUE_TYPE,
)

print(
    f"History: {history['reported'].min().date()} "
    f"-> {history['reported'].max().date()}"
)
print(
    f"Available {FORECAST_YEAR} actuals: "
    f"{actual_2026['reported'].min().date()} "
    f"-> {actual_2026['reported'].max().date()}"
)
print(f"Actual reports available: {actual_2026['reports'].sum():,.0f}")

History: 2020-01-02 -> 2025-12-31
Available 2026 actuals: 2026-01-01 -> 2026-06-08
Actual reports available: 3,795


## 3. Fully recursive historical backtest

Pretend the backtest year is completely unknown. No actual values from that year are fed into lags.

In [3]:
backtest_rows = []
backtest_forecasts = {}

for trend_mode in BACKTEST_TREND_MODES:
    metrics, forecast, model = recursive_backtest(
        history,
        BACKTEST_YEAR,
        model_name=SELECTED_MODEL,
        trend_mode=trend_mode,
        include_holidays=INCLUDE_HOLIDAYS,
        log_divisor=LOG_TREND_DIVISOR,
    )

    backtest_rows.append({
        "trend": trend_mode,
        **metrics.to_dict(),
        "alpha": float(model.params["alpha"]),
    })
    backtest_forecasts[trend_mode] = forecast

backtest_results = pd.DataFrame(backtest_rows).set_index("trend")
display(backtest_results.round(3))

,MAE,RMSE,Mean error,Actual total,Predicted total,Total difference,alpha
trend,,,,,,,
linear,7.915,10.733,-1.766,10147.0,10791.500,644.500,0.09
log,7.585,10.201,1.096,10147.0,9747.085,-399.915,0.09


## 4. 2026: daily-updated vs fully recursive

Daily-updated predictions use real preceding 2026 counts. The fully recursive forecast uses no 2026 actual counts after 2025-12-31.

In [4]:
model_2026 = fit_nb(
    history,
    model_name=SELECTED_MODEL,
    trend_mode=TREND_MODE,
    include_holidays=INCLUDE_HOLIDAYS,
)

origin = history["reported"].min()

daily_updated = daily_updated_predictions(
    model_2026,
    history[["reported", "reports"]],
    actual_2026,
    origin=origin,
    log_divisor=LOG_TREND_DIVISOR,
)

full_dates = pd.date_range(
    f"{FORECAST_YEAR}-01-01",
    FORECAST_END,
    freq="D",
)

fully_recursive = recursive_forecast(
    model_2026,
    history[["reported", "reports"]],
    full_dates,
    origin=origin,
    log_divisor=LOG_TREND_DIVISOR,
)

recursive_available = fully_recursive[
    fully_recursive["reported"] <= actual_2026["reported"].max()
]

hybrid_full, hybrid_updated, hybrid_future = hybrid_forecast(
    model_2026,
    history[["reported", "reports"]],
    actual_2026,
    FORECAST_END,
    origin=origin,
    log_divisor=LOG_TREND_DIVISOR,
)

comparison = pd.DataFrame({
    "Fully recursive": forecast_metrics(actual_2026, recursive_available),
    "Daily updated": forecast_metrics(actual_2026, daily_updated),
}).T

display(comparison.round(3))

print("\nFULL-YEAR FORECAST TOTALS")
print(f"Fully recursive:          {fully_recursive['predicted'].sum():,.1f}")
print(f"Daily-updated + recursive:{hybrid_full['predicted'].sum():,.1f}")
print(f"NB alpha:                 {model_2026.params['alpha']:.4f}")

,MAE,RMSE,Mean error,Actual total,Predicted total,Total difference
Fully recursive,7.035,10.098,-0.729,3795.0,3910.891,115.891
Daily updated,6.314,8.919,-0.252,3795.0,3835.121,40.121



FULL-YEAR FORECAST TOTALS
Fully recursive:          11,268.8
Daily-updated + recursive:11,191.5
NB alpha:                 0.0892


## 5. Interpretation

- **Daily updated** is the realistic operational mode: tomorrow's lag features use real observations already received.
- **Fully recursive** is the harder planning mode: future predicted means must feed future lag features.
- Keep these two use cases separate when reporting model quality.